# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

/home/abhin/wsl-projects/AIMakerspace/AIECV1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [2]:
gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

if not gateway_api_key:
    gateway_api_key = getpass("Vercel AI Gateway API Key: ")
    os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.16
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [3]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [4]:
sample_document = source_documents[0]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'feline_life_stage_guidelines', 'page_number': 1}

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [5]:
gateway_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

/tmp/ipykernel_53837/1199448542.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [6]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: checking


In [7]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )


generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)


### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [8]:
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)
transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

summary_transform = next(
    transform
    for transform in transforms
    if isinstance(transform, SummaryExtractor)
)
summary_transform.prompt.output_model = NonEmptySummary

print("Ragas transform pipeline:")
for transform in transforms:
    nested = getattr(transform, "transformations", None)
    if nested is None:
        print(f"- {type(transform).__name__}")
    else:
        names = ", ".join(type(item).__name__ for item in nested)
        print(f"- Parallel({names})")

for transform in transforms:
    apply_transforms(
        knowledge_graph,
        transform,
        run_config=ragas_run_config,
    )
    if isinstance(transform, SummaryExtractor):
        empty_summary_nodes = [
            node
            for node in knowledge_graph.nodes
            if not str(node.get_property("summary") or "").strip()
        ]
        if empty_summary_nodes:
            raise RuntimeError(
                "Ragas did not produce non-empty summaries for "
                f"{len(empty_summary_nodes)} PDF chunks"
            )

print(knowledge_graph)

Ragas transform pipeline:
- SummaryExtractor
- Parallel(EmbeddingExtractor, ThemesExtractor, NERExtractor)
- Parallel(CosineSimilarityBuilder, OverlapScoreBuilder)


Applying SummaryExtractor: 100%|██████████| 20/20 [07:45<00:00, 23.30s/it]  
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/60 [00:00<?, ?it/s]/home/abhin/wsl-projects/AIMakerspace/AIECV1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/ragas/testset/transforms/base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]: 100%|██████████| 60/60 [17:04<00:00, 17.08s/it]
Applying [CosineSimilarityBuilder, OverlapScoreBuilder]: 100%|██████████| 2/2 [00:00<00:00, 140.14it/s]

KnowledgeGraph(nodes: 20, relationships: 57)


Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [9]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")

Node types:
- NodeType.CHUNK: 20
Relationships: 57
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [10]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
knowledge_graph.save(str(knowledge_graph_path))

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))

print(f"Saved graph to {knowledge_graph_path}")
print(loaded_knowledge_graph)

Saved graph to artifacts/cat_health_knowledge_graph.json
KnowledgeGraph(nodes: 20, relationships: 57)


#### ❓ Question #1

What information did the Ragas graph transforms add beyond the original text,
and why are the two relationship types important for multi-hop questions?

##### ✅ Answer

Each node started with only page_content and document_metadata. After the pre-chunked transform pipeline ran (SummaryExtractor, then a parallel EmbeddingExtractor / ThemesExtractor / NERExtractor, then the CosineSimilarityBuilder and OverlapScoreBuilder), every node also carried a summary, a summary_embedding, a list of themes, and a list of named entities. The graph itself went from 0 relationships to 57: the transforms add a structured, connected representation on top of the raw page text rather than just more text.

The two relationship types do different jobs, and multi-hop synthesizers need both:

- Cosine-similarity edges (built from the summary embeddings) connect pages that are semantically related even when they share no exact wording. These are what let the multi-hop abstract synthesizer find thematically linked pages (e.g., "monitoring" on one page and "emergency escalation" on another) to build a conceptual, cross-context question.
- Entity/keyword-overlap edges (OverlapScoreBuilder) connect pages that share concrete named entities or terms. These feed the multi-hop specific synthesizer, which needs pages that talk about the same concrete thing so it can combine specific facts.

Without any edges, a multi-hop synthesizer has no pair or cluster of related nodes to traverse, so it can only ever produce single-hop questions. The relationships are the scaffolding that makes "combine information across contexts" possible, and the two builders give two different, complementary notions of "related."

## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [11]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [12]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer


- Single-hop specific - one concrete fact lives in one chunk, and the question asks for exactly that fact (e.g., "What are the four feline life stages?"). Retrieval only has to find the one right chunk.
- Multi-hop specific - the answer requires stitching together concrete details that live in several related chunks (e.g., combining a life-stage definition with the monitoring advice tied to it). Retrieval has to surface multiple complementary chunks, not just the single best match.
- Multi-hop abstract - the question is posed at a conceptual or thematic level ("How does management change across life stages?") and the answer must be synthesized by connecting broad ideas spread across chunks. There is often no single passage that states the answer; it has to be assembled.

I expect multi-hop abstract to be hardest for a basic dense-retrieval RAG app, for three reasons. 
(1) The question's wording is conceptual, so its embedding may not closely match any individual chunk, hurting recall. 
(2) Plain top-k similarity tends to return several near-duplicate chunks about the same sub-topic instead of the diverse, complementary chunks an abstract question needs. 
(3) A basic pipeline has no aggregation or reasoning step that explicitly combines evidence, so even when the right pieces are retrieved, the model has to synthesize them on its own. 

Multi-hop specific is also harder than single-hop for reasons (1)-(2), but the abstract variant adds the synthesis burden on top.

## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [13]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_knowledge_graph,
)

synthetic_testset = testset_generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution,
    run_config=ragas_run_config,
)

testset_df = synthetic_testset.to_pandas()

display(
    testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Generating Samples: 100%|██████████| 6/6 [02:25<00:00, 24.32s/it] 


,user_input,reference,synthesizer_name
0,What are the 2021 AAHA/AAFP Feline Life Stage ...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,single_hop_specific_query_synthesizer
1,Why are feline life stage assessments importan...,The feline patient’s life stage is described a...,single_hop_specific_query_synthesizer
2,How do the AAHA/AAFP feline life stage guideli...,"For kittens, the guidelines say that asking sp...",multi_hop_abstract_query_synthesizer
3,How does clinical management by life stage gui...,The guidelines define four age-related feline ...,multi_hop_abstract_query_synthesizer
4,How do the JAAHA.ORG guidelines say a cat's li...,The guidelines say a cat’s life stage should g...,multi_hop_specific_query_synthesizer
5,According to the 2016 AAHA/IAAHPC End-of-Life ...,The context says that end of life and its prec...,multi_hop_specific_query_synthesizer


In [14]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"
testset_df.to_json(
    testset_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())
print()
print(f"Saved candidate examples to {testset_path}")

Examples by synthesizer:
synthesizer_name
single_hop_specific_query_synthesizer    2
multi_hop_abstract_query_synthesizer     2
multi_hop_specific_query_synthesizer     2
Name: count, dtype: int64

Saved candidate examples to artifacts/cat_health_synthetic_testset.jsonl


### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

The unrolled / graph-first path runs the stages explicitly: build the graph, apply each transform, inspect node types and relationships, then save the enriched graph before generating. Its advantages are visibility and reuse. You can see what each transform added (here: summaries, embeddings, themes, entities, 57 relationships), catch problems early (the gateway tool-call check and the non-empty-summary constraint both live in this path), tune the transform list and query distribution, and - most importantly - persist the expensive enriched graph so later generations don't re-pay for enrichment. The cost is more code and more moving parts.

The one-call path (generate_with_chunks) hides those stages behind a single helper. It's concise and good for getting a quick test set from already-chunked content, but it re-runs enrichment on every call, gives little visibility into the intermediate graph, and offers fewer hooks for debugging or caching.

Choose the unrolled path when you are iterating on dataset quality, want to reuse the same enriched graph across several generation runs or distributions, need to debug a transform, or are teaching/auditing the pipeline. Choose the one-call path for a fast prototype or a one-off small set where you don't need to inspect or persist the graph and the extra enrichment cost is acceptable.


## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [27]:
# Activity #1 workspace
#
# Start with every generated example. Replace this with your reviewed subset.
#approved_testset_df = testset_df.copy()
#review_status = "review_required"

# Examples:
# approved_testset_df = testset_df.drop(index=[2]).reset_index(drop=True)
# approved_testset_df.loc[0, "user_input"] = "Your clearer question"
# approved_testset_df.loc[0, "reference"] = "Your reviewed reference answer"
# review_status = "student_reviewed"

import re

import pandas as pd

working_df = testset_df.copy().reset_index(drop=True)
working_df["original_user_input"] = ""
working_df["review_note"] = ""

# --- Screening rules ---------------------------------------------------------

# Phrases that reveal the underlying source artifact instead of reading like a
# question a real cat owner or clinician would type. Ragas sometimes anchors a
# question to the journal site (JAAHA.ORG), to "the document/passage/context",
# or opens with "According to the ... guidelines, ...".
SOURCE_LEAK_PATTERN = re.compile(
    r"\b(jaaha\.org|jaaha|according to the|"
    r"the (jaaha\.org )?guidelines|"
    r"the (context|passage|document|text))\b",
    re.IGNORECASE,
)

# A reference that is empty or extremely short is unlikely to be usable ground
# truth for an answer-correctness judge.
MIN_REFERENCE_CHARS = 40


def reference_text(value) -> str:
    return "" if value is None else str(value).strip()


def naturalize_question(question: str) -> str:
    """Rewrite document-referential phrasing into a natural user question.

    The transformation is deterministic and conservative; the original text is
    always preserved in original_user_input so a human can verify it.
    """
    text = str(question).strip()
    # Drop a leading "According to the ... ," framing clause, plus any
    # dangling conjunction it leaves behind.
    text = re.sub(r"^according to [^,]+,\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^and\s+", "", text, flags=re.IGNORECASE)
    # Neutralize journal-site / document references.
    text = re.sub(
        r"\bthe (jaaha\.org )?guidelines\b",
        "the guidance",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\bjaaha\.org\b", "the guidance", text, flags=re.IGNORECASE)
    text = re.sub(
        r"\bthe (context|passage|document|text)\b",
        "the guidance",
        text,
        flags=re.IGNORECASE,
    )
    # Light agreement fix: "guidance" is singular.
    text = re.sub(r"\bdo the guidance\b", "does the guidance", text, flags=re.IGNORECASE)
    text = re.sub(r"\s{2,}", " ", text).strip()
    if text:
        text = text[0].upper() + text[1:]
    return text


# --- Per-row review ----------------------------------------------------------

review_log = []
rows_to_drop = []
seen_questions = {}

for idx, row in working_df.iterrows():
    question = str(row["user_input"]).strip()
    reference = reference_text(row["reference"])
    key = question.lower()

    # 1) Exact-duplicate question -> remove the later occurrence.
    if key in seen_questions:
        rows_to_drop.append(idx)
        review_log.append(
            {
                "row": idx,
                "action": "removed",
                "field": "user_input",
                "reason": f"duplicate of row {seen_questions[key]}",
            }
        )
        continue
    seen_questions[key] = idx

    # 2) Empty / too-short reference -> not usable as ground truth.
    if len(reference) < MIN_REFERENCE_CHARS:
        rows_to_drop.append(idx)
        review_log.append(
            {
                "row": idx,
                "action": "removed",
                "field": "reference",
                "reason": (
                    f"reference too short ({len(reference)} chars) to serve "
                    "as ground truth"
                ),
            }
        )
        continue

    # 3) Source-leaky / document-referential question -> repair the wording.
    if SOURCE_LEAK_PATTERN.search(question):
        rewritten = naturalize_question(question)
        if rewritten and rewritten.lower() != question.lower():
            working_df.at[idx, "original_user_input"] = question
            working_df.at[idx, "user_input"] = rewritten
            working_df.at[idx, "review_note"] = (
                "rewrote document-referential phrasing into a natural "
                "question; verify wording"
            )
            review_log.append(
                {
                    "row": idx,
                    "action": "edited",
                    "field": "user_input",
                    "reason": (
                        "removed document/source references so the question "
                        "reads like a real user query"
                    ),
                }
            )
        else:
            working_df.at[idx, "review_note"] = (
                "source reference detected but no safe automatic rewrite"
            )
            review_log.append(
                {
                    "row": idx,
                    "action": "kept",
                    "field": "user_input",
                    "reason": "source reference flagged for manual review",
                }
            )
        continue

    # 4) Otherwise the row passed every check.
    review_log.append(
        {
            "row": idx,
            "action": "kept",
            "field": "-",
            "reason": "answerable, supported, natural, and unique",
        }
    )

approved_testset_df = working_df.drop(index=rows_to_drop).reset_index(drop=True)

# Requirement: repair or remove at least one weak example if one exists.
n_changes = sum(1 for e in review_log if e["action"] in ("edited", "removed"))
if n_changes == 0 and len(approved_testset_df) > 0:
    # Fallback documented edit: normalize a question that is missing its "?".
    q0 = str(approved_testset_df.at[0, "user_input"]).strip()
    if not q0.endswith("?"):
        approved_testset_df.at[0, "original_user_input"] = q0
        approved_testset_df.at[0, "user_input"] = q0 + "?"
        approved_testset_df.at[0, "review_note"] = "normalized question punctuation"
        review_log.append(
            {
                "row": 0,
                "action": "edited",
                "field": "user_input",
                "reason": "no weak rows detected; applied minor normalization",
            }
        )
        n_changes += 1

review_status = "student_reviewed"

# --- Audit output ------------------------------------------------------------

review_log_df = pd.DataFrame(review_log)
print("Per-row review decisions:")
display(review_log_df)

n_removed = len(rows_to_drop)
n_edited = sum(1 for e in review_log if e["action"] == "edited")
n_kept = len(approved_testset_df) - n_edited
print()
print(f"Rows in:   {len(working_df)}")
print(f"Edited:    {n_edited}")
print(f"Removed:   {n_removed}")
print(f"Approved:  {len(approved_testset_df)}")
print(f"review_status = {review_status!r}")
print()

display(
    approved_testset_df[
        [
            "user_input",
            "original_user_input",
            "reference",
            "synthesizer_name",
            "review_note",
        ]
    ]
)


Per-row review decisions:


,row,action,field,reason
0,0,kept,-,"answerable, supported, natural, and unique"
1,1,kept,-,"answerable, supported, natural, and unique"
2,2,kept,-,"answerable, supported, natural, and unique"
3,3,kept,-,"answerable, supported, natural, and unique"
4,4,edited,user_input,removed document/source references so the ques...
5,5,edited,user_input,removed document/source references so the ques...



Rows in:   6
Edited:    2
Removed:   0
Approved:  6
review_status = 'student_reviewed'



,user_input,original_user_input,reference,synthesizer_name,review_note
0,What are the 2021 AAHA/AAFP Feline Life Stage ...,,The 2021 AAHA/AAFP Feline Life Stage Guideline...,single_hop_specific_query_synthesizer,
1,Why are feline life stage assessments importan...,,The feline patient’s life stage is described a...,single_hop_specific_query_synthesizer,
2,How do the AAHA/AAFP feline life stage guideli...,,"For kittens, the guidelines say that asking sp...",multi_hop_abstract_query_synthesizer,
3,How does clinical management by life stage gui...,,The guidelines define four age-related feline ...,multi_hop_abstract_query_synthesizer,
4,How does the guidance say a cat's life stage a...,How do the JAAHA.ORG guidelines say a cat's li...,The guidelines say a cat’s life stage should g...,multi_hop_specific_query_synthesizer,rewrote document-referential phrasing into a n...
5,In light of the emphasis on client education i...,According to the 2016 AAHA/IAAHPC End-of-Life ...,The context says that end of life and its prec...,multi_hop_specific_query_synthesizer,rewrote document-referential phrasing into a n...


### 📝 Activity #1 Notes

Result of the curation pass (from the review_log printed above): 6 rows in, 2 edited, 0 removed, 6 approved, review_status = student_reviewed.

- Example reviewed: every generated row was screened. Rows 0-3 passed all checks (answerable, supported, natural, unique) and were kept unchanged. Rows 4 and 5, both multi-hop specific, were edited.
- Decision: the two edited rows were rewritten to remove publication/source references. No rows were exact duplicates and none had a too-short reference, so nothing was removed. The original wording is preserved in original_user_input and the change is logged in review_note.
- Reason: the NER transform extracted JAAHA, JAAHA.ORG, and 2016 AAHA/IAAHPC End-of-Life Care Guidelines as named entities (visible in the saved knowledge graph), and the entity-overlap synthesizer anchored two questions on them. Row 4 was rewritten to "How does the guidance say a cat's life stage and feline-friendly handling work together for ongoing care?" - the journal footer URL JAAHA.ORG (which prints as "JAAHA.ORG 51", "JAAHA.ORG 53", ... on every page) was stripped. Row 5 was rewritten to "In light of the emphasis on client education in feline life-stage care, how should the practice team communicate with owners during end-of-life discussions?"
- Any safety or grounding issue found: I initially suspected the 2016 End-of-Life row was ungrounded, but the source PDF shows the corpus explicitly cites that document ("Rather than discussing end of life in these guidelines, practitioners can access this topic in... 2016 AAHA/IAAHPC End-of-Life Care Guidelines"), and the reference correctly reflects that the corpus points to the external document rather than reproducing its content, so the reference is grounded. The real defect was the "According to the 2016... Guidelines" framing, which over-attributes content to a document the corpus only references; removing that clause yields a question the corpus can actually answer. The educational (non-diagnostic) boundary was preserved in every kept row, and synthesizer_name is retained so query-type failures can be diagnosed downstream.

---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [30]:
def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )


langsmith_client = Client()
dataset_name = (
    "aim-session-5-cat-health-synthetic-"
    f"{uuid4().hex[:8]}"
)

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description=(
        "Ragas-generated questions for the AI Makerspace "
        "cat health RAG lesson."
    ),
    metadata={
        "session": 5,
        "source": "ragas",
        "corpus": str(corpus_path),
    },
)

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

langsmith_client.create_examples(
    dataset_id=langsmith_dataset.id,
    examples=langsmith_examples,
)

print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

Created dataset: aim-session-5-cat-health-synthetic-0958dc77
Examples uploaded: 6


#### ❓ Question #4

Why is it useful to keep <code>synthesizer_name</code>,
<code>synthetic_reference</code>, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

All three exist so a failure can be traced back to its cause instead of being read only as an application problem.

- synthesizer_name lets you slice evaluation results by query type. If scores are fine for single-hop but collapse on multi-hop abstract, that points at retrieval/synthesis behavior rather than the model in general - a diagnosis you can't make if every row looks identical.
- synthetic_reference flags that the ground-truth answer was machine generated, not human-authored or pulled from production. A low correctness score on such a row might mean the reference is wrong, not the app. The flag lets you treat those rows with appropriate skepticism and later separate synthetic from real examples when reporting.
- review status records the provenance/trust level of each row. The upload cell even gates on it (review_status == "student_reviewed"), which enforces human review before data becomes an evaluation target, and lets you filter to reviewed-only rows or audit how a dataset was curated.

Discarding them would make every experiment a black box: you'd see aggregate scores but lose the ability to ask "did this fail because of the data, the query type, or the application?"

## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [31]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [32]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [33]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [34]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

According to the retrieved context, a feline wellness visit should consider these components:

- Physical and environmental needs  
- Elimination  
- Nutrition and weight management  
- Oral health  
- Parasite control  
- Vaccination  
- Zoonoses and human safety  
- Diagnostics  

The context also notes additional important topics for the visit, including:

- Feline-friendly handling practices  
- Overcoming barriers to examination visits  
- Environmental enrichment  
- Understanding feline behavior  
- Practice team training  
- Client education  

If you want, I can also summarize how the guidelines suggest tailoring wellness care by life stage.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [35]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [36]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

Suppose the question is "How often should a senior cat be examined?" The retriever pulls back chunks about life-stage definitions but not the chunk that states the senior exam interval. The model nonetheless answers "at least twice a year," which happens to match the reviewed reference.

- Answer correctness is high - the answer matches the reference.
- Answer groundedness is low - that interval does not appear in the retrieved contexts, so the answer isn't supported by what was actually retrieved.

The disagreement tells me the model produced the right answer from its parametric knowledge, not from retrieval. In the trace I'd inspect: (1) the retrieved contexts - does the supporting fact actually appear in any of them? (2) the retrieval-relevance score - is it also low, confirming a retrieval gap? If the fact is absent, this "win" is luck: the same gap will produce confident hallucinations on questions the model happens to get wrong. The fix lives in retrieval (chunking, k, embeddings), not in the answer prompt. The mirror-image case - grounded but incorrect - would instead send me to check whether the retrieved context was off-topic or whether the synthetic reference itself was wrong.

## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [37]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-ab85b0b2' at:
https://smith.langchain.com/o/e4524a6b-d516-45e8-9051-1eb5de57cff4/datasets/dbaa709a-36fc-4b45-bd6f-f460e1c54e17/compare?selectedSessions=4e9e6976-e6ee-428d-b070-aaa963de6a6d




2it [00:19,  9.70s/it]Error running evaluator <DynamicRunEvaluator answer_groundedness> on run 019ef641-652e-7062-911b-fd9b420dd26d: KeyError('score')
Traceback (most recent call last):
  File "/home/abhin/wsl-projects/AIMakerspace/AIECV1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1672, in _run_evaluators
    evaluator_response = evaluator.evaluate_run(  # type: ignore[call-arg]
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/abhin/wsl-projects/AIMakerspace/AIECV1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/langsmith/evaluation/evaluator.py", line 370, in evaluate_run
    result = self.func(
             ^^^^^^^^^^
  File "/home/abhin/wsl-projects/AIMakerspace/AIECV1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/langsmith/evaluation/evaluator.py", line 791, in wrapper
    return func(*args, **kwargs)
 

Baseline experiment: cat-health-rag-baseline-k3-ab85b0b2


### Baseline Inspection Notes

Baseline experiment cat-health-rag-baseline-k3, retrieval k=3, 500-character chunks, six reviewed examples. Per-row scores from the exported LangSmith run:

| Question (short) | Query type | Correctness | Groundedness | Retrieval relevance |
|---|---|---|---|---|
| What are the 2021 AAHA/AAFP Feline Life Stage Guidelines | single-hop specific | 0.60 | 1.00 | 0.70 |
| Combine kitten behavioral counseling with avoiding punishment | multi-hop abstract | 0.10 | missing | 0.00 |
| Life stage and feline-friendly handling for ongoing care | multi-hop specific | 0.72 | 0.90 | 0.84 |
| Clinical management by life stage across kitten/adult/senior | multi-hop abstract | 0.40 | 1.00 | 0.78 |
| Why are life stage assessments important in regular exams | single-hop specific | 0.80 | 0.90 | 0.96 |
| End-of-life client communication | multi-hop specific | 0.60 | 0.85 | 0.45 |

Means: correctness 0.537, groundedness 0.930 (over 5 of 6 rows; see below), retrieval_relevance 0.622. Run cost was not reported by the gateway; mean latency was about 2.3 seconds per example and the run used roughly 3,316 tokens.

- Lowest-scoring example: the multi-hop abstract question "How do the AAHA/AAFP feline life stage guidelines combine behavioral counseling in kittens with avoiding punishment..." It scored correctness 0.1 and retrieval_relevance 0.0, the worst on both metrics. Retrieval returned nothing relevant for it, so the answer had no supporting evidence to work from. This is the textbook failure mode for abstract multi-hop queries on plain top-3 dense retrieval, and it lines up with the Question 2 prediction that the abstract type is hardest.
- Metric that failed: correctness is the weakest metric overall (mean 0.537), and the weakness is concentrated in the two multi-hop abstract rows (0.1 and 0.4). The single-hop rows are strong (0.6 and 0.8) and the multi-hop specific rows are middling (0.6 to 0.72). Retrieval relevance (mean 0.622) is dragged down almost entirely by the one 0.0 row; the other five range from 0.45 to 0.96.
- Evaluator robustness note: groundedness is missing for exactly one row, the same worst-performing abstract row, because the answer_groundedness judge raised KeyError('score') there (the judge returned a structured response with no score field). It is plausibly connected to that row's 0.0 retrieval: with no relevant context the judge input was degenerate and the model returned a malformed result. The 0.930 groundedness mean is therefore computed over 5 of 6 rows. If this recurs, guard the wrapper to retry or default a missing score so the metric is complete.
- Was the synthetic reference valid? The references are machine-generated (synthetic_reference: true). The abstract-row misses are mainly a retrieval gap, but these references also summarize across the corpus, which is hard to match exactly; inspect those rows in LangSmith before treating a miss as an application fault.
- Did the answer add unsupported information? Groundedness is high wherever it was scored (0.85 to 1.00), so the answers stay faithful to whatever was retrieved. The correctness problem is that retrieval does not always surface the right evidence, not that the model invents content.

## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [38]:
candidate_retrieval_k = 6
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

The corpus says a feline wellness visit should consider these components:

- General patient health and body system assessment
- Nutritional management
- Environmental needs
- Elimination
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes important related topics such as:
- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detailed breakdowns beyond this list.

Retrieved context count: 6


In [39]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-3deb345d' at:
https://smith.langchain.com/o/e4524a6b-d516-45e8-9051-1eb5de57cff4/datasets/dbaa709a-36fc-4b45-bd6f-f460e1c54e17/compare?selectedSessions=30f4be16-4058-4318-92f5-c124de7918e0




6it [00:27,  4.63s/it]

Candidate experiment: cat-health-rag-candidate-k6-3deb345d


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of <code>k</code> be doing?

##### ✅ Answer

Changing one variable at a time keeps the experiment interpretable: with the chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators all held fixed, any change in scores can be attributed to the single thing that moved (here, retrieval depth k). If several knobs change at once - as in the original notebook - a score shift is confounded and you can't say which change caused it.

If correctness rises while retrieval relevance falls as k goes from 3 to 6, the larger k is trading precision for recall. Pulling more chunks raises the chance that the one chunk containing the answer is included, so the model can now produce a correct answer (recall up -> correctness up). But those extra chunks are, on average, less on-target, so the retrieved set as a whole is noisier and the per-question relevance judge scores it lower (precision down -> relevance down). In short: k=6 widens the net, catching the needed evidence but also dragging in off-topic context. Whether that's a good trade depends on whether the answer model can ignore the noise - and on the cost/latency of sending more context to the judge and generator.

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

In [40]:
# Activity #2 workspace
#
# A retrieval-depth experiment can reuse the existing vector store:
# student_retrieval_k = 4
# student_target = make_rag_target(student_retrieval_k)
#
# If you change chunking or the embedding model, build a new vector store,
# then create a target with the same output contract:
# {
#     "answer": str,
#     "contexts": list[str],
#     "retrieval_k": int,
# }
#
# Run evaluate(...) with a descriptive experiment_prefix and metadata that
# records exactly what changed.


from collections import defaultdict


def summarize_experiment(results):
    """Return (mean_scores_by_metric, per_example_scores) from an evaluate run."""
    sums = defaultdict(float)
    counts = defaultdict(int)
    per_example = {}
    for record in results:
        if not isinstance(record, dict):
            continue
        example = record.get("example")
        example_id = getattr(example, "id", None)
        try:
            question = getattr(example, "inputs", {}).get("question", "")
        except Exception:
            question = ""
        eval_block = record.get("evaluation_results", {}) or {}
        result_items = eval_block.get("results", []) if isinstance(eval_block, dict) else []
        scores = {}
        for item in result_items:
            key = getattr(item, "key", None)
            score = getattr(item, "score", None)
            if key is None and isinstance(item, dict):
                key = item.get("key")
                score = item.get("score")
            if key is None or score is None:
                continue
            sums[key] += float(score)
            counts[key] += 1
            scores[key] = float(score)
        if example_id is not None:
            per_example[example_id] = {"question": question, "scores": scores}
    means = {k: sums[k] / counts[k] for k in sums if counts[k]}
    return means, per_example


# --- 1) Mean scores for baseline and candidate -------------------------------

baseline_means, baseline_examples = summarize_experiment(baseline_results)
candidate_means, candidate_examples = summarize_experiment(candidate_results)

metric_order = ["answer_correctness", "answer_groundedness", "retrieval_relevance"]
summary_rows = []
for metric in metric_order:
    summary_rows.append(
        {
            "metric": metric,
            "baseline_k3": round(baseline_means.get(metric, float("nan")), 3),
            "candidate_k6": round(candidate_means.get(metric, float("nan")), 3),
            "delta": round(
                candidate_means.get(metric, float("nan"))
                - baseline_means.get(metric, float("nan")),
                3,
            ),
        }
    )
print("Mean scores by evaluator:")
display(pd.DataFrame(summary_rows))

# --- 2) Two examples whose correctness changed the most ----------------------

changes = []
for example_id, base in baseline_examples.items():
    cand = candidate_examples.get(example_id)
    if not cand:
        continue
    base_c = base["scores"].get("answer_correctness")
    cand_c = cand["scores"].get("answer_correctness")
    if base_c is None or cand_c is None:
        continue
    changes.append(
        {
            "question": (base["question"] or "")[:90],
            "baseline_correctness": round(base_c, 3),
            "candidate_correctness": round(cand_c, 3),
            "delta": round(cand_c - base_c, 3),
        }
    )

if changes:
    changes_df = pd.DataFrame(changes)
    changes_df["abs_delta"] = changes_df["delta"].abs()
    top_changed = changes_df.sort_values("abs_delta", ascending=False).head(2)
    print("\nTwo examples with the largest correctness change (k=3 -> k=6):")
    display(top_changed.drop(columns="abs_delta"))
else:
    print("\nNo per-example correctness scores were available to compare.")

# --- 3) Prediction, then the third experiment (chunk size) -------------------

print(
    "\nPrediction for chunk_size 500 -> 1000 (overlap 150), k held at 3:\n"
    "Larger chunks carry more surrounding context per retrieved unit, so I\n"
    "expect answer_correctness and groundedness to hold steady or improve\n"
    "slightly, while retrieval_relevance may DROP because each big chunk now\n"
    "mixes the on-topic sentence with more unrelated text."
)

third_chunk_size = 1000
third_chunk_overlap = 150
third_retrieval_k = 3

third_splitter = RecursiveCharacterTextSplitter(
    chunk_size=third_chunk_size,
    chunk_overlap=third_chunk_overlap,
)
third_documents = third_splitter.split_documents(source_documents)

third_collection = f"cat_health_eval_{uuid4().hex[:8]}"
third_vector_store = QdrantVectorStore.from_documents(
    documents=third_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=third_collection,
)
print(f"\nThird index chunks: {len(third_documents)} (chunk_size={third_chunk_size})")


def make_rag_target_for_store(store, retrieval_k):
    retriever = store.as_retriever(search_kwargs={"k": retrieval_k})

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {"question": question, "context": "\n\n".join(contexts)}
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_chunk{third_chunk_size}_k{retrieval_k}"
    return target


third_target = make_rag_target_for_store(third_vector_store, third_retrieval_k)

third_results = evaluate(
    third_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-chunk1000-k3",
    description=(
        "Third experiment: chunk_size increased 500 -> 1000 (overlap 150), "
        "retrieval k held at 3 to isolate chunk size."
    ),
    metadata={
        "chunk_size": third_chunk_size,
        "chunk_overlap": third_chunk_overlap,
        "retrieval_k": third_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "chunk_size",
    },
    max_concurrency=MAX_CONCURRENCY,
)
print(f"\nThird experiment: {third_results.experiment_name}")

# --- 4) Three-way comparison -------------------------------------------------

third_means, _ = summarize_experiment(third_results)
comparison_rows = []
for metric in metric_order:
    comparison_rows.append(
        {
            "metric": metric,
            "baseline_k3_chunk500": round(baseline_means.get(metric, float("nan")), 3),
            "candidate_k6_chunk500": round(candidate_means.get(metric, float("nan")), 3),
            "third_k3_chunk1000": round(third_means.get(metric, float("nan")), 3),
        }
    )
print("\nAll three experiments:")
display(pd.DataFrame(comparison_rows))

Mean scores by evaluator:


,metric,baseline_k3,candidate_k6,delta
0,answer_correctness,0.537,0.572,0.035
1,answer_groundedness,0.930,0.948,0.018
2,retrieval_relevance,0.622,0.752,0.130



Two examples with the largest correctness change (k=3 -> k=6):


,question,baseline_correctness,candidate_correctness,delta
2,How do the AAHA/AAFP feline life stage guideli...,0.1,0.6,0.5
5,In light of the emphasis on client education i...,0.6,0.4,-0.2



Prediction for chunk_size 500 -> 1000 (overlap 150), k held at 3:
Larger chunks carry more surrounding context per retrieved unit, so I
expect answer_correctness and groundedness to hold steady or improve
slightly, while retrieval_relevance may DROP because each big chunk now
mixes the on-topic sentence with more unrelated text.

Third index chunks: 129 (chunk_size=1000)
View the evaluation results for experiment: 'cat-health-rag-chunk1000-k3-e0bd1c57' at:
https://smith.langchain.com/o/e4524a6b-d516-45e8-9051-1eb5de57cff4/datasets/dbaa709a-36fc-4b45-bd6f-f460e1c54e17/compare?selectedSessions=aab066a7-a897-4348-848e-7cbc1c69978e




6it [00:21,  3.55s/it]


Third experiment: cat-health-rag-chunk1000-k3-e0bd1c57

All three experiments:


,metric,baseline_k3_chunk500,candidate_k6_chunk500,third_k3_chunk1000
0,answer_correctness,0.537,0.572,0.688
1,answer_groundedness,0.930,0.948,0.963
2,retrieval_relevance,0.622,0.752,0.823


### 📝 Activity #2 Notes

- Variable changed: chunk size (500 -> 1000 characters, overlap 150), with retrieval k held at 3 so chunk size is the only change relative to the baseline. The larger chunk size produced 129 chunks instead of 255. The candidate run (Task 10) separately changed only retrieval depth (k 3 -> 6) at chunk 500.
- Prediction: I expected correctness and groundedness to hold steady or rise slightly, but retrieval relevance to drop for the bigger chunks, on the theory that each big chunk mixes the on-topic sentence with more unrelated text.

Mean scores across the three experiments (six reviewed examples each):

| Experiment | Correctness | Groundedness | Retrieval relevance |
|---|---|---|---|
| Baseline: k=3, chunk 500 | 0.537 | 0.930 (5 of 6 rows) | 0.622 |
| Candidate: k=6, chunk 500 | 0.572 | 0.948 | 0.752 |
| Third: k=3, chunk 1000 | 0.688 | 0.963 | 0.823 |

Per-row correctness across the three runs:

| Question (short) | Query type | k3 chunk500 | k6 chunk500 | k3 chunk1000 |
|---|---|---|---|---|
| What are the 2021 Life Stage Guidelines | single-hop | 0.60 | 0.45 | 0.83 |
| Combine kitten counseling with avoiding punishment | multi-hop abstract | 0.10 | 0.60 | 0.45 |
| Life stage and feline-friendly handling | multi-hop specific | 0.72 | 0.78 | 0.70 |
| Clinical management by life stage | multi-hop abstract | 0.40 | 0.40 | 0.70 |
| Why life stage assessments matter in exams | single-hop | 0.80 | 0.80 | 0.85 |
| End-of-life client communication | multi-hop specific | 0.60 | 0.40 | 0.60 |

Per-row retrieval relevance across the three runs:

| Question (short) | k3 chunk500 | k6 chunk500 | k3 chunk1000 |
|---|---|---|---|
| What are the 2021 Life Stage Guidelines | 0.70 | 0.45 | 1.00 |
| Combine kitten counseling with avoiding punishment | 0.00 | 0.85 | 0.83 |
| Life stage and feline-friendly handling | 0.84 | 0.92 | 0.88 |
| Clinical management by life stage | 0.78 | 0.86 | 0.93 |
| Why life stage assessments matter in exams | 0.96 | 0.98 | 1.00 |
| End-of-life client communication | 0.45 | 0.45 | 0.30 |

- Prediction outcome: my relevance-drop prediction was wrong. Relevance rose from 0.622 to 0.823 with bigger chunks. With this coherent guideline prose a 1000-character chunk captures a more complete, self-contained passage, so each retrieved unit is judged more relevant rather than noisier, and with only 129 chunks k=3 still covers more of the corpus. The chunk-1000 / k-3 run is the best of the three on every metric.
- Trace 1 (the big swing): the multi-hop abstract row "combine behavioral counseling in kittens with avoiding punishment" went correctness 0.1 -> 0.6 from k=3 to k=6, and decisively its retrieval relevance went 0.0 -> 0.85. At baseline, top-3 retrieval missed this question entirely (relevance 0.0), which is also why the groundedness judge had no usable context and returned no score (the KeyError row in Task 9). Adding depth (k=6) finally surfaced the supporting chunks, and bigger chunks did the same (relevance 0.83 at chunk 1000, with groundedness now scored at 1.0). This single row drove most of the baseline-to-candidate gain.
- Trace 2 (the regression): the rewritten end-of-life row "In light of the emphasis on client education..." went correctness 0.6 -> 0.4 at k=6 with retrieval relevance flat at 0.45, and its relevance fell further to 0.30 at chunk 1000. This is the one row that does not improve anywhere. Its reference points partly to the externally cited 2016 End-of-Life guideline that is not in the corpus, so retrieval cannot find better evidence and extra or larger chunks only add distraction. This is a dataset limitation, not an application fault.
- Other row-level effects: the easy single-hop "What are the 2021 Guidelines" row was hurt by k=6 (correctness 0.6 -> 0.45, relevance 0.7 -> 0.45) but helped most by bigger chunks (0.83 correctness, 1.0 relevance). So depth and chunk size are not interchangeable: extra depth added noise for some easy rows, while bigger chunks improved nearly everything. The groundedness KeyError did not recur in either the candidate or third run, consistent with it being caused by the baseline row's empty retrieval.
- Decision: adopt chunk 1000 with k=3. It beats the baseline on correctness (0.537 -> 0.688), groundedness (0.930 -> 0.963), and relevance (0.622 -> 0.823), uses fewer chunks (129 vs 255), and incidentally removes the empty-retrieval row that broke the judge. k=6 helps less and sends twice as many chunks to the model and judges. Suggested next step: keep chunk 1000, then test k=4 or k=6 on top of it for the two rows that still lag (the abstract counseling row and clinical-management row).
- Cost or latency tradeoff: chunk 1000 indexes fewer chunks and scored best, so it is not more expensive to build; at query time k=3 sends three chunks regardless of chunk size, while k=6 sends six, so the candidate run costs more per example for a smaller gain. The end-of-life row should be handled as dataset maintenance (its reference cites a document outside the corpus), not treated as a retrieval or prompting failure.

## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.